In [ ]:
# Install required packages if they are missing
required_packages <- c("cellWise", "robustHD", "ggplot2", "ggrepel", "dplyr", "gridExtra", "tidyr")
new_packages <- required_packages[!(required_packages %in% installed.packages()[,"Package"])]
if(length(new_packages)) install.packages(new_packages)

In [ ]:
# =============================================================================
# 01_simulation_study.R
#
# Partial replication of the simulation study from:
#   Hubert et al. (2019) MacroPCA, Technometrics 61, 459-473.
#
# We replicate:
#   - Figure 7: 20% NAs + 20% cellwise outliers  (MSE vs gamma)
#   - Figure 9: 20% NAs + 10% cellwise + 10% rowwise outliers (MSE vs gamma)
#
# Methods compared: ICPCA, MROBPCA, MacroPCA  (as in the paper)
# Metric: MSE against baseline PCA on clean data  (paper Section 5)
# Data generating process: A09 covariance, n=100, d=200, k=6  (paper Section 5)
#
# NOTE: With d=200 this takes ~20-40 min. Reduce n_sim or d to prototype faster.
# =============================================================================

library(cellWise)   # MacroPCA, ICPCA, MROBPCA, DDC
library(MASS)       # mvrnorm (backup)
library(ggplot2)
library(dplyr)
library(tidyr)

set.seed(2024)

# =============================================================================
# 1. Data generating process  (Section 5 of the paper)
# =============================================================================

n      <- 100   # observations
d      <- 200   # variables
k      <- 6     # true number of components
n_sim  <- 30    # Monte Carlo replications (paper uses 100; reduce for speed)

cat("Building A09 covariance matrix (d =", d, ")...\n")

# A09 structured correlation: rho_{ij} = (-0.9)^|i-j|
idx    <- matrix(1:d, d, d)
R_A09  <- (-0.9)^abs(idx - t(idx))

# Target eigenvalues: 6 large + 194 small (paper Section 5)
lambda_large  <- c(30, 25, 20, 15, 10, 5)
lambda_small  <- seq(0.098, 0.0015, length.out = d - k)
lambda_target <- c(lambda_large, lambda_small)

# Build Sigma by replacing eigenvalues of R_A09
eig_R     <- eigen(R_A09, symmetric = TRUE)
Sigma     <- eig_R$vectors %*% diag(lambda_target) %*% t(eig_R$vectors)
Sigma     <- (Sigma + t(Sigma)) / 2          # ensure symmetry

# Variables for contamination
sigma_j   <- sqrt(diag(Sigma))               # column SDs, used for cellwise shift
v_kp1     <- eig_R$vectors[, k + 1]          # (k+1)-th eigenvector, for rowwise shift

# Fast clean-data generator using Cholesky
chol_Sig  <- chol(Sigma)                     # upper-triangular
generate_clean <- function(n) {
  matrix(rnorm(n * d), n, d) %*% chol_Sig   # n x d
}

cat("Done. Sigma built.\n")

# =============================================================================
# 2. MSE helper
#
# Baseline: classical PCA on the CLEAN rows of the uncontaminated data X0.
# For each method applied to contaminated data, compute predictions for those
# same clean rows and measure MSE against the baseline predictions.
# Paper eq: MSE = (1/cd) * sum_{i in C} sum_j (xhat_ij - xhat^C_ij)^2
# =============================================================================
compute_predictions <- function(center, loadings, X_data) {
  # X_data: n x d (may contain NAs; missing entries get predicted from subspace)
  # Returns n x d matrix of predicted values
  X_c  <- sweep(X_data, 2, center, "-")      # centre
  # For rows with NAs, use only observed entries to compute scores
  scores <- matrix(NA, nrow(X_data), ncol(loadings))
  for (i in seq_len(nrow(X_data))) {
    obs <- !is.na(X_c[i, ])
    if (sum(obs) >= ncol(loadings)) {
      # Least-squares projection onto observed dimensions
      P_obs    <- loadings[obs, , drop = FALSE]
      scores[i, ] <- solve(t(P_obs) %*% P_obs) %*% t(P_obs) %*% X_c[i, obs]
    } else {
      scores[i, ] <- 0
    }
  }
  Xhat <- sweep(scores %*% t(loadings), 2, center, "+")
  Xhat
}

compute_mse <- function(Xhat_method, Xhat_base, C_rows) {
  # MSE over clean rows C and all d columns
  diff  <- Xhat_method[C_rows, ] - Xhat_base[C_rows, ]
  mean(diff^2, na.rm = TRUE)
}

# =============================================================================
# 3. Contamination functions
# =============================================================================
add_nas <- function(X, frac = 0.20) {
  Xc     <- X
  n_miss <- round(frac * length(X))
  idx    <- sample(length(X), n_miss)
  Xc[idx] <- NA
  Xc
}

add_cellwise <- function(X, frac = 0.20, gamma) {
  Xc     <- X
  n_cont <- round(frac * length(X))
  idx    <- sample(length(X), n_cont)
  # shift by gamma * sigma_j  (paper: replace x_ij with gamma * sigma_j)
  col_idx <- ((idx - 1) %% d) + 1
  Xc[idx] <- gamma * sigma_j[col_idx]
  Xc
}

add_rowwise <- function(X, frac = 0.20, gamma) {
  Xc      <- X
  bad     <- sample(nrow(X), round(frac * nrow(X)))
  # shift from N(gamma * v_{k+1}, Sigma)  (paper Section 5)
  shift   <- gamma * v_kp1                   # d-vector
  for (i in bad) {
    noise      <- matrix(rnorm(d), 1, d) %*% chol_Sig
    Xc[i, ]    <- shift + noise
  }
  list(X = Xc, bad_rows = bad)
}

# =============================================================================
# 4. Run simulation for a given scenario
# Scenario A: 20% NA + 20% cellwise  (Figure 7)
# Scenario B: 20% NA + 10% cellwise + 10% rowwise  (Figure 9)
# =============================================================================
gamma_vals <- c(0, 1, 2, 3, 5, 7, 10, 15, 20)

run_scenario <- function(scenario_name, frac_cell, frac_row) {
  cat("\n=== Scenario:", scenario_name, "===\n")

  results <- expand.grid(
    gamma  = gamma_vals,
    method = c("ICPCA", "MROBPCA", "MacroPCA"),
    mse    = NA_real_,
    stringsAsFactors = FALSE
  )

  for (gi in seq_along(gamma_vals)) {
    gamma <- gamma_vals[gi]
    cat("  gamma =", gamma, "\n")

    mse_icpca    <- numeric(n_sim)
    mse_mrobpca  <- numeric(n_sim)
    mse_macro    <- numeric(n_sim)

    for (s in seq_len(n_sim)) {
      # --- Generate clean data ---
      X0 <- generate_clean(n)

      # --- Baseline: classical PCA on full clean data ---
      pca_base    <- prcomp(X0, center = TRUE, scale. = FALSE)
      center_base <- colMeans(X0)
      scores_base <- X0 %*% pca_base$rotation[, 1:k, drop = FALSE]
      Xhat_base   <- sweep(
        scores_base %*% t(pca_base$rotation[, 1:k, drop = FALSE]),
        2, center_base, "+"
      )

      # --- Contaminate ---
      X_cont  <- X0
      bad_rows <- integer(0)

      if (frac_cell > 0) {
        X_cont <- add_cellwise(X_cont, frac = frac_cell, gamma = gamma)
      }
      if (frac_row > 0) {
        rw       <- add_rowwise(X_cont, frac = frac_row, gamma = gamma)
        X_cont   <- rw$X
        bad_rows <- rw$bad_rows
      }
      # Add NAs last (so we know which cells are outliers vs missing)
      X_cont <- add_nas(X_cont, frac = 0.20)

      # Clean rows for MSE evaluation
      C_rows <- setdiff(seq_len(n), bad_rows)

      # --- ICPCA ---
      fit_icpca <- tryCatch(
        ICPCA(X_cont, k = k),
        error = function(e) NULL
      )
      if (!is.null(fit_icpca)) {
        Xhat_i  <- compute_predictions(fit_icpca$center,
                                       fit_icpca$loadings, X0)
        mse_icpca[s] <- compute_mse(Xhat_i, Xhat_base, C_rows)
      } else {
        mse_icpca[s] <- NA
      }

      # --- MROBPCA ---
      fit_mrob <- tryCatch(
        MROBPCA(X_cont, k = k),
        error = function(e) NULL
      )
      if (!is.null(fit_mrob)) {
        Xhat_m  <- compute_predictions(fit_mrob$center,
                                       fit_mrob$loadings, X0)
        mse_mrobpca[s] <- compute_mse(Xhat_m, Xhat_base, C_rows)
      } else {
        mse_mrobpca[s] <- NA
      }

      # --- MacroPCA ---
      fit_macro <- tryCatch(
        MacroPCA(X_cont, k = k),
        error = function(e) NULL
      )
      if (!is.null(fit_macro)) {
        Xhat_p  <- compute_predictions(fit_macro$center,
                                       fit_macro$loadings, X0)
        mse_macro[s] <- compute_mse(Xhat_p, Xhat_base, C_rows)
      } else {
        mse_macro[s] <- NA
      }
    } # end sim loop

    results$mse[results$gamma == gamma & results$method == "ICPCA"]    <- mean(mse_icpca,   na.rm = TRUE)
    results$mse[results$gamma == gamma & results$method == "MROBPCA"]  <- mean(mse_mrobpca, na.rm = TRUE)
    results$mse[results$gamma == gamma & results$method == "MacroPCA"] <- mean(mse_macro,   na.rm = TRUE)
  } # end gamma loop

  results
}

# Run both scenarios
res_fig7 <- run_scenario("Fig7: 20% NA + 20% cellwise",
                         frac_cell = 0.20, frac_row = 0.00)

res_fig9 <- run_scenario("Fig9: 20% NA + 10% cell + 10% row",
                         frac_cell = 0.10, frac_row = 0.10)

# =============================================================================
# 5. Plots  (line plots of avg MSE vs gamma, one curve per method)
# =============================================================================
method_colors <- c(
  "ICPCA"    = "#E07B54",
  "MROBPCA"  = "#4C8BB5",
  "MacroPCA" = "#2CA02C"
)
method_lines <- c(
  "ICPCA"    = "dashed",
  "MROBPCA"  = "dotted",
  "MacroPCA" = "solid"
)

make_mse_plot <- function(results, title_str, y_lim = NULL) {
  p <- ggplot(results, aes(x = gamma, y = mse,
                           colour = method, linetype = method)) +
    geom_line(linewidth = 0.9) +
    geom_point(size = 2) +
    scale_colour_manual(values = method_colors) +
    scale_linetype_manual(values = method_lines) +
    labs(
      title    = title_str,
      subtitle = paste0("n = ", n, ", d = ", d, ", k = ", k,
                        ", ", n_sim, " replications, A09 covariance"),
      x        = expression(gamma ~ "(contamination distance)"),
      y        = "Average MSE",
      colour   = "Method",
      linetype = "Method"
    ) +
    theme_bw(base_size = 13) +
    theme(legend.position = "bottom")

  if (!is.null(y_lim)) p <- p + coord_cartesian(ylim = y_lim)
  p
}

p_fig7 <- make_mse_plot(
  res_fig7,
  "Figure 7 replication: 20% missing + 20% cellwise outliers"
)

p_fig9 <- make_mse_plot(
  res_fig9,
  "Figure 9 replication: 20% missing + 10% cellwise + 10% rowwise outliers"
)

print(p_fig7)
print(p_fig9)

ggsave("sim_figure7_replication.pdf", p_fig7, width = 7, height = 5)
ggsave("sim_figure9_replication.pdf", p_fig9, width = 7, height = 5)

# =============================================================================
# 6. Combined panel (both scenarios side by side)
# =============================================================================
res_fig7$scenario <- "20% NA + 20% cellwise"
res_fig9$scenario <- "20% NA + 10% cell + 10% row"
res_combined      <- rbind(res_fig7, res_fig9)

p_combined <- ggplot(res_combined,
                     aes(x = gamma, y = mse,
                         colour = method, linetype = method)) +
  geom_line(linewidth = 0.9) +
  geom_point(size = 1.8) +
  scale_colour_manual(values = method_colors) +
  scale_linetype_manual(values = method_lines) +
  facet_wrap(~ scenario, scales = "free_y") +
  labs(
    title    = "Simulation study: ICPCA vs MROBPCA vs MacroPCA",
    subtitle = paste0("A09 covariance, n = ", n, ", d = ", d,
                      ", k = ", k, ", ", n_sim, " MC replications"),
    x        = expression(gamma),
    y        = "Average MSE",
    colour   = "Method",
    linetype = "Method"
  ) +
  theme_bw(base_size = 12) +
  theme(legend.position = "bottom")

print(p_combined)
ggsave("sim_combined_panel.pdf", p_combined, width = 11, height = 5)

cat("\n=== Simulation study complete. Saved three PDFs. ===\n")

# Print summary tables
cat("\n--- Figure 7 results (avg MSE) ---\n")
print(pivot_wider(res_fig7[, c("gamma","method","mse")],
                  names_from = method, values_from = mse))

cat("\n--- Figure 9 results (avg MSE) ---\n")
print(pivot_wider(res_fig9[, c("gamma","method","mse")],
                  names_from = method, values_from = mse))


In [ ]:
# =============================================================================
# 02_real_data_analysis.R
#
# Replication of Section 3 (and partially Section 4) of:
#   Hubert et al. (2019) MacroPCA, Technometrics 61, 459-473.
#
# Dataset: Top Gear cars (297 cars x 11 continuous variables)
# Reproduces: Figure 3 (residual maps), Figure 4 (outlier maps),
#             Figure 5 (online prediction)
# =============================================================================

library(cellWise)
library(robustHD)
library(ggplot2)
library(ggrepel)
library(dplyr)
library(tidyr)       # needed for pivot_longer / pivot_wider
library(gridExtra)

# =============================================================================
# 1. Load Top Gear data  — robust multi-fallback approach
#    (some Kaggle installs of robustHD omit the bundled dataset)
# =============================================================================
topgear <- NULL

# Attempt 1 – standard call
tryCatch({
  data("topgear", package = "robustHD", envir = environment())
  topgear <- get("topgear", envir = environment())
  cat("topgear loaded via data()\n")
}, error = function(e) invisible(NULL))

# Attempt 2 – capitalised variant used in older versions
if (is.null(topgear)) {
  tryCatch({
    data("TopGear", package = "robustHD", envir = environment())
    topgear <- get("TopGear", envir = environment())
    cat("topgear loaded as TopGear\n")
  }, error = function(e) invisible(NULL))
}

# Attempt 3 – scan the package data directory directly
if (is.null(topgear)) {
  pkg_data <- system.file("data", package = "robustHD")
  rda_files <- list.files(pkg_data,
    pattern = "(?i)topgear", full.names = TRUE, perl = TRUE)
  if (length(rda_files) > 0) {
    env_tmp <- new.env()
    load(rda_files[1], envir = env_tmp)
    nm <- ls(env_tmp)
    if (length(nm) > 0) {
      topgear <- get(nm[1], envir = env_tmp)
      cat("topgear loaded from package data dir:", rda_files[1], "\n")
    }
  }
}

# Attempt 4 – reinstall robustHD and retry
if (is.null(topgear)) {
  message("topgear not found — reinstalling robustHD from CRAN...")
  install.packages("robustHD", repos = "https://cloud.r-project.org",
                   quiet = TRUE)
  library(robustHD)
  tryCatch({
    data("topgear", package = "robustHD", envir = environment())
    topgear <- get("topgear", envir = environment())
    cat("topgear loaded after reinstall\n")
  }, error = function(e) stop("Could not load topgear even after reinstall."))
}

if (is.null(topgear)) stop("topgear dataset could not be loaded by any method.")

# =============================================================================
# 2. Inspect and preprocess
# =============================================================================
car_names <- rownames(topgear)

cat("\n=== Top Gear Dataset: ", nrow(topgear), "rows x", ncol(topgear), "cols ===\n")

# The 11 continuous variables used in the paper
cont_vars <- c("Price", "Displacement", "BHP", "Torque",
               "Acceleration", "TopSpeed", "MPG",
               "Weight", "Length", "Width", "Height")

available <- intersect(cont_vars, colnames(topgear))
missing_v <- setdiff(cont_vars, colnames(topgear))
if (length(missing_v) > 0)
  cat("NOTE — variables not found (check column names):", missing_v, "\n")

X_raw <- as.matrix(topgear[, available, drop = FALSE])
cat("Using", ncol(X_raw), "variables for", nrow(X_raw), "cars.\n")
cat("Existing missing cells:", sum(is.na(X_raw)),
    sprintf("(%.1f%%)\n\n", 100 * mean(is.na(X_raw))))

# =============================================================================
# 3. Log-transform five skewed variables  (paper Section 3)
# =============================================================================
log_vars <- intersect(c("Price", "Displacement", "BHP", "Torque", "TopSpeed"),
                      colnames(X_raw))
X <- X_raw
for (v in log_vars) X[, v] <- log(X_raw[, v])
X[!is.finite(X)] <- NA  # replace Inf/NaN from log(0) or log(negative)
cat("Log-transformed:", paste(log_vars, collapse = ", "), "\n\n")

# =============================================================================
# 4 + 5. Fit MacroPCA first (it may drop rows), then ICPCA on the aligned data
# =============================================================================
k <- 2    # k = 2 as in paper

# =============================================================================
# 5. Fit MacroPCA
# =============================================================================
cat("Fitting MacroPCA (k =", k, ")...\n")
fit_macro <- MacroPCA(X, k = k)
# FIX: Convert 1D indcells output into a proper 2D matrix
macro_ind_mat <- matrix(0L, nrow(fit_macro$stdResid), ncol(fit_macro$stdResid))
macro_ind_mat[fit_macro$stdResid > 2.576 & !is.na(fit_macro$stdResid)] <- 1L
macro_ind_mat[fit_macro$stdResid < -2.576 & !is.na(fit_macro$stdResid)] <- -1L
fit_macro$indcells <- macro_ind_mat

# ALIGN: MacroPCA silently drops rows that are entirely NA.
# Re-index car_names and X to only the rows that survived.
kept_rows  <- match(rownames(fit_macro$stdResid), rownames(X))
car_names  <- car_names[kept_rows]
X          <- X[kept_rows, ]

# Save copy with NAs for ICPCA residual map (to know which cells were NA)
X_with_na <- X

# Impute all NAs in X with column medians for ICPCA
for (j in seq_len(ncol(X))) {
  col_med <- median(X[, j], na.rm = TRUE)
  if (!is.finite(col_med)) col_med <- 0
  X[is.na(X[, j]), j] <- col_med
}

# Now fit ICPCA on the aligned (295-row) data so all outputs share the same row index
cat("Fitting ICPCA (k =", k, ")...\n")
set.seed(42)
X_icpca <- X
for (j in seq_len(ncol(X_icpca))) { col_med <- median(X_icpca[, j], na.rm=TRUE); if (!is.finite(col_med)) col_med <- 0; X_icpca[is.na(X_icpca[, j]), j] <- col_med; if (sd(X_icpca[, j]) < 1e-10) X_icpca[, j] <- X_icpca[, j] + rnorm(nrow(X_icpca), 0, 1e-6) }
fit_icpca <- tryCatch(
  ICPCA(X_icpca, k = k),
  error = function(e) { message("ICPCA failed: ", conditionMessage(e)); NULL }
)
if (is.null(fit_icpca)) stop("ICPCA could not be fitted — check for Inf/NaN in X.")
cat("ICPCA: Cumulative variance explained:",
    round(cumsum(fit_icpca$eigenvalues /
                   sum(fit_icpca$eigenvalues))[1:k], 3), "\n\n")

cat("MacroPCA: Cumulative variance explained:",
    round(cumsum(fit_macro$eigenvalues /
                   sum(fit_macro$eigenvalues))[1:k], 3), "\n")
cat("MacroPCA: Flagged cellwise outliers:",
    sum(fit_macro$indcells != 0, na.rm = TRUE), "\n")
cat("MacroPCA: Flagged casewise outliers:",
    sum(fit_macro$indrows,        na.rm = TRUE), "\n\n")

# =============================================================================
# 6. Select 24 representative cars for the residual map  (paper Figure 3)
# =============================================================================
notable_cars <- c(
  "Bugatti Veyron", "Pagani Huayra",
  "BMW i3", "Chevrolet Volt", "Vauxhall Ampera", "Mitsubishi i-MiEV",
  "Renault Twizy", "Citroen DS5",
  "Land Rover Defender", "Mercedes-Benz G",
  "Ssangyong Rodius"
)

note_idx <- which(car_names %in% notable_cars)
top_od   <- order(fit_macro$OD, decreasing = TRUE)[1:10]
sel_rows <- unique(c(note_idx, top_od))
sel_rows <- sel_rows[seq_len(min(24, length(sel_rows)))]
sel_names <- car_names[sel_rows]

cat("Selected", length(sel_rows), "cars for residual map.\n")

# =============================================================================
# 7. MacroPCA residual map  (Figure 3 right)
# =============================================================================
# stdResid is the paper's standardized residual matrix R_{n,d}
pdf("residual_map_macropca.pdf", width = 10, height = 7)
cellMap(
  fit_macro$stdResid[sel_rows, ],
  indcells     = (fit_macro$indcells != 0)[sel_rows, ],
  rowlabels    = sel_names,
  columnlabels = colnames(X),
  mTitle       = "MacroPCA \u2013 Residual map (Figure 3 right)"
)
dev.off()
cat("Saved: residual_map_macropca.pdf\n")

# =============================================================================
# 8. ICPCA residual map  (Figure 3 left)
#    ICPCA has no built-in cell map; we build standardised residuals manually
# =============================================================================

# Predictions from ICPCA
Xhat_icpca <- sweep(
  fit_icpca$scores %*% t(fit_icpca$loadings),
  2, fit_icpca$center, "+"
)

# NA-imputed X: start from original (with NAs), fill NAs with fitted values
X_naimp_icpca <- X_with_na
for (j in seq_len(ncol(X_with_na))) {
  na_j <- is.na(X_with_na[, j])
  X_naimp_icpca[na_j, j] <- Xhat_icpca[na_j, j]
}

resid_icpca <- X_naimp_icpca - Xhat_icpca
col_mad     <- apply(resid_icpca, 2, function(x) mad(x, na.rm = TRUE))
col_mad[col_mad < 1e-10] <- 1
stdR_icpca  <- sweep(resid_icpca, 2, col_mad, "/")

# Flag cells beyond ±2.576  (= sqrt(qchisq(0.99, 1)), same threshold as paper)
indcells_icpca <- matrix(0L, nrow(X), ncol(X))
indcells_icpca[!is.na(stdR_icpca) & stdR_icpca >  2.576] <-  1L
indcells_icpca[!is.na(stdR_icpca) & stdR_icpca < -2.576] <- -1L

pdf("residual_map_icpca.pdf", width = 10, height = 7)
cellMap(
  stdR_icpca[sel_rows, ],
  indcells     = (indcells_icpca != 0)[sel_rows, ],
  rowlabels    = sel_names,
  columnlabels = colnames(X),
  mTitle       = "ICPCA \u2013 Residual map (Figure 3 left)"
)
dev.off()
cat("Saved: residual_map_icpca.pdf\n")

# =============================================================================
# 9. Outlier map helper  (Figure 4)
# =============================================================================
make_outlier_map <- function(SD, OD, cSD, cOD, labels,
                             title_str, label_these = NULL) {
  type <- dplyr::case_when(
    SD > cSD & OD > cOD  ~ "Bad leverage point",
    SD > cSD & OD <= cOD ~ "Good leverage point",
    SD <= cSD & OD > cOD ~ "Orthogonal outlier",
    TRUE                  ~ "Regular"
  )
  df <- data.frame(SD, OD, type, label = labels,
                   stringsAsFactors = FALSE)
  if (is.null(label_these)) label_these <- labels[type != "Regular"]
  df$show_label <- df$label %in% label_these

  ggplot(df, aes(x = SD, y = OD, colour = type)) +
    geom_point(aes(shape = type), size = 1.8, alpha = 0.75) +
    geom_vline(xintercept = cSD, linetype = "dashed", colour = "grey50") +
    geom_hline(yintercept = cOD, linetype = "dashed", colour = "grey50") +
    geom_text_repel(
      data = subset(df, show_label),
      aes(label = label), size = 2.8, max.overlaps = 20, segment.size = 0.3
    ) +
    scale_colour_manual(values = c(
      "Regular"             = "#AAAAAA",
      "Good leverage point" = "#4C8BB5",
      "Orthogonal outlier"  = "#E07B54",
      "Bad leverage point"  = "#D62728"
    )) +
    scale_shape_manual(values = c(
      "Regular"             = 1,
      "Good leverage point" = 2,
      "Orthogonal outlier"  = 16,
      "Bad leverage point"  = 17
    )) +
    labs(title = title_str,
         x     = "Score distance (SD)",
         y     = "Orthogonal distance (OD)",
         colour = NULL, shape = NULL) +
    theme_bw(base_size = 12) +
    theme(legend.position = "bottom")
}

cars_to_label <- c(
  "BMW i3", "Bugatti Veyron", "Pagani Huayra",
  "Vauxhall Ampera", "Chevrolet Volt", "Renault Twizy",
  "Citroen DS5", "Mitsubishi i-MiEV",
  "Land Rover Defender", "Mercedes-Benz G"
)

# MacroPCA outlier map
p_om_macro <- make_outlier_map(
  SD          = fit_macro$SD,
  OD          = fit_macro$OD,
  cSD         = fit_macro$cutoffSD,
  cOD         = fit_macro$cutoffOD,
  labels      = car_names,
  title_str   = "MacroPCA \u2013 Outlier map (Figure 4 right)",
  label_these = cars_to_label
)

# ICPCA outlier map — compute SD/OD manually
OD_icpca <- sqrt(rowSums((X_naimp_icpca - Xhat_icpca)^2, na.rm = TRUE))
SD_icpca <- sqrt(rowSums(
  sweep(fit_icpca$scores^2, 2, fit_icpca$eigenvalues, "/")
))
cSD_icpca <- sqrt(qchisq(0.99, df = k))
# OD cutoff: use ICPCA output if present, else MCD-based approximation
cOD_icpca <- if (!is.null(fit_icpca$cutoffOD)) {
  fit_icpca$cutoffOD
} else {
  od23 <- OD_icpca^(2/3)
  (median(od23, na.rm = TRUE) +
     mad(od23, na.rm = TRUE) * qnorm(0.99))^(3/2)
}

p_om_icpca <- make_outlier_map(
  SD          = SD_icpca,
  OD          = OD_icpca,
  cSD         = cSD_icpca,
  cOD         = cOD_icpca,
  labels      = car_names,
  title_str   = "ICPCA \u2013 Outlier map (Figure 4 left)",
  label_these = cars_to_label
)

p_outlier_panel <- gridExtra::grid.arrange(p_om_icpca, p_om_macro, ncol = 2)
ggsave("outlier_map_panel.pdf", p_outlier_panel, width = 14, height = 6)
cat("Saved: outlier_map_panel.pdf\n")

# =============================================================================
# 10. Loadings comparison bar chart
# =============================================================================
load_df <- data.frame(
  variable    = colnames(X),
  PC1_MacroPCA = fit_macro$loadings[, 1],
  PC2_MacroPCA = fit_macro$loadings[, 2],
  PC1_ICPCA    = fit_icpca$loadings[, 1],
  PC2_ICPCA    = fit_icpca$loadings[, 2],
  stringsAsFactors = FALSE
)

# Pivot: split on LAST underscore to handle "MacroPCA" correctly
load_long <- load_df %>%
  pivot_longer(-variable, names_to = "key", values_to = "loading") %>%
  mutate(
    PC     = sub("^(PC[0-9]+)_.*$", "\\1", key),
    method = sub("^PC[0-9]+_(.*)$",  "\\1", key)
  )

p_loadings <- ggplot(load_long,
                     aes(x = variable, y = loading, fill = method)) +
  geom_bar(stat = "identity", position = "dodge", alpha = 0.85, width = 0.7) +
  geom_hline(yintercept = 0, colour = "grey30", linewidth = 0.4) +
  scale_fill_manual(values = c("MacroPCA" = "#2CA02C", "ICPCA" = "#E07B54")) +
  facet_wrap(~ PC, ncol = 1) +
  labs(title = "Loadings: ICPCA vs MacroPCA (Top Gear, k = 2)",
       x = NULL, y = "Loading", fill = "Method") +
  theme_bw(base_size = 12) +
  theme(axis.text.x  = element_text(angle = 45, hjust = 1),
        legend.position = "bottom")

ggsave("loadings_comparison.pdf", p_loadings, width = 8, height = 7)
cat("Saved: loadings_comparison.pdf\n")

# =============================================================================
# 11. Online prediction demo  (Section 4 / Figure 5)
#     Fit MacroPCA on remaining cars, predict the selected cars out-of-sample
# =============================================================================
cat("\n=== Online prediction (Section 4 / Figure 5) ===\n")

all_rows   <- seq_len(nrow(X))
train_rows <- setdiff(all_rows, sel_rows)
d_cols     <- ncol(X)

cat("Training on", length(train_rows), "cars, predicting", length(sel_rows), "\n")

# --- Refit MacroPCA on training data, with fallbacks ---
fit_train <- NULL

# Strategy 1: use imputed X with relaxed DDC parameters
tryCatch({
  X_tr <- X[train_rows, ]
  rownames(X_tr) <- seq_len(nrow(X_tr))
  fit_train <- MacroPCA(X_tr, k = k,
    MacroPCApars = list(DDCpars = list(numDiscrete = 0, fracNA = 1.0)))
  cat("  Training fit succeeded (Strategy 1: imputed data, relaxed DDCpars)\n")
}, error = function(e) {
  cat("  Strategy 1 failed:", conditionMessage(e), "\n")
})

# Strategy 2: use original data with NAs + relaxed DDC
if (is.null(fit_train)) {
  tryCatch({
    X_tr2 <- X_with_na[train_rows, ]
    rownames(X_tr2) <- seq_len(nrow(X_tr2))
    fit_train <- MacroPCA(X_tr2, k = k,
      MacroPCApars = list(DDCpars = list(numDiscrete = 0, fracNA = 1.0)))
    cat("  Training fit succeeded (Strategy 2: NAs + relaxed DDCpars)\n")
  }, error = function(e) {
    cat("  Strategy 2 failed:", conditionMessage(e), "\n")
  })
}

# Strategy 3: fall back to the full-data fit
if (is.null(fit_train)) {
  cat("  Using full-data MacroPCA fit as fallback (Strategy 3)\n")
  fit_train <- fit_macro
}

# --- Predict each test car ---
X_test <- X[sel_rows, , drop = FALSE]
rownames(X_test) <- seq_len(nrow(X_test))

pred_list <- lapply(seq_len(nrow(X_test)), function(i) {
  tryCatch(
    MacroPCApredict(
      Xnew            = X_test[i, , drop = FALSE],
      InitialMacroPCA = fit_train
    ),
    error = function(e) {
      # silent fallback
      NULL
    }
  )
})

n_ok <- sum(!sapply(pred_list, is.null))
cat("  Predictions succeeded for", n_ok, "of", nrow(X_test), "test cars\n")

# --- Collect results ---
stdR_pred <- do.call(rbind, lapply(pred_list, function(r) {
  if (!is.null(r) && !is.null(r$stdResid)) as.numeric(r$stdResid)
  else rep(NA_real_, d_cols)
}))

indcells_pred <- matrix(0L, nrow = length(pred_list), ncol = d_cols)
for (i in seq_along(pred_list)) {
  r <- pred_list[[i]]
  if (!is.null(r) && !is.null(r$indcells)) {
    ic <- r$indcells
    if (length(ic) == d_cols) {
      indcells_pred[i, ] <- as.integer(ic != 0)
    } else if (length(ic) > 0 && length(ic) < d_cols) {
      valid <- ic[ic >= 1 & ic <= d_cols]
      if (length(valid) > 0) indcells_pred[i, valid] <- 1L
    }
  }
}
colnames(stdR_pred) <- colnames(X)
rownames(stdR_pred) <- sel_names

if (n_ok > 0 && any(!is.na(stdR_pred))) {
  pdf("online_prediction_comparison.pdf", width = 14, height = 7)
  par(mfrow = c(1, 2))
  cellMap(
    fit_macro$stdResid[sel_rows, ],
    indcells = (fit_macro$indcells != 0)[sel_rows, ],
    rowlabels = sel_names, columnlabels = colnames(X),
    mTitle = "In-sample (all cars fitted)"
  )
  cellMap(
    stdR_pred, indcells = (indcells_pred != 0),
    rowlabels = sel_names, columnlabels = colnames(X),
    mTitle = "Out-of-sample (selected cars predicted)"
  )
  dev.off()
  cat("Saved: online_prediction_comparison.pdf\n")
} else {
  cat("NOTE: MacroPCApredict returned no usable results — skipping Figure 5 plot.\n")
}

# =============================================================================
# 12. Flagged cars summary table
# =============================================================================
macro_type <- dplyr::case_when(
  fit_macro$SD > fit_macro$cutoffSD & fit_macro$OD > fit_macro$cutoffOD ~
    "Bad leverage point",
  fit_macro$SD > fit_macro$cutoffSD  ~ "Good leverage point",
  fit_macro$OD > fit_macro$cutoffOD  ~ "Orthogonal outlier",
  TRUE                               ~ "Regular"
)

flagged_df <- data.frame(
  Car    = car_names,
  SD     = round(fit_macro$SD, 2),
  OD     = round(fit_macro$OD, 2),
  Type   = macro_type,
  stringsAsFactors = FALSE
) %>%
  filter(Type != "Regular") %>%
  arrange(desc(OD))

cat("\n=== MacroPCA: Flagged cars (non-regular) ===\n")
print(flagged_df)

cat("\n=== Real data analysis complete ===\n")


# =============================================================================
# 13. Online prediction stability study (colleague's idea)
#
#     Idea: fit MacroPCA on cumulatively-nested training subsets
#       100% -> 90% -> 80% -> 70% -> 60% -> 50%
#     where the 10% held out at 90% is contained in the 20% held out at 80%,
#     etc.  Pick 5 cars from the FIRST 10% held out and use MacroPCApredict
#     at every training size.  Measure how the predicted scores drift from
#     the "ground truth" scores produced by the 100% fit.
#
#     Distance measure:
#       absolute  = || t_hat - t_ref ||_2
#       relative  = || t_hat - t_ref ||_2 / || t_ref ||_2
#
#     Note on sign convention: PCA loadings are unique only up to sign
#     flips of each axis, so we sign-align each subset fit against the
#     100% reference before computing distances.
# =============================================================================
cat("\n=== Online prediction stability vs. training-set size ===\n")

set.seed(42)

# IMPORTANT: use X_with_na throughout — MacroPCA handles NAs natively.
# The median-imputed X causes DDC's MAD check to flag all columns as
# "zero or tiny MAD" on any subset, because imputed medians are identical.
# car_names has already been trimmed to the 295 rows MacroPCA kept.
n      <- nrow(X_with_na)
Xstab  <- X_with_na    # 295 x 11, original NAs preserved

perm   <- sample(n)
fracs  <- c(1.00, 0.90, 0.80, 0.70, 0.60, 0.50)

# Reference: full-data MacroPCA fit
fit_ref <- fit_macro
k_ref   <- ncol(fit_ref$loadings)
P_ref   <- fit_ref$loadings

# Pick 5 test cars from the first 10% removed
holdout_first10 <- perm[seq_len(floor(0.10 * n))]
n_test          <- min(5, length(holdout_first10))
test_idx        <- holdout_first10[seq_len(n_test)]
test_names      <- car_names[test_idx]   # car_names is length 295
cat("Test cars (held out from every non-100% fit):\n")
cat(paste(test_names, collapse=", "), "\n")

# Reference scores: predict each test car using the 100%-fit model
ref_pred <- lapply(test_idx, function(i) {
  tryCatch(MacroPCApredict(Xnew = Xstab[i, , drop=FALSE],
                           InitialMacroPCA = fit_ref),
           error = function(e) NULL)
})
T_ref <- do.call(rbind, lapply(ref_pred, function(r)
  if (!is.null(r) && !is.null(r$T)) as.numeric(r$T)
  else rep(NA_real_, k_ref)))
rownames(T_ref) <- test_names
colnames(T_ref) <- paste0("PC", seq_len(k_ref))

cat("\nReference scores (from 100% fit):\n")
print(round(T_ref, 3))

# Sign-alignment helper (PCA axes unique only up to sign flip)
align_signs <- function(P_sub, P_ref) {
  k <- min(ncol(P_sub), ncol(P_ref))
  sapply(seq_len(k), function(j) {
    s <- sum(P_sub[, j] * P_ref[, j])
    if (is.na(s) || s == 0) 1L else sign(s)
  })
}

# Loop over training fractions
results <- list()

for (frac in fracs) {
  n_train   <- floor(frac * n)
  # Cumulatively-nested: the held-out set grows as frac decreases.
  # Training = LAST n_train rows of perm; held-out = FIRST (n-n_train).
  train_idx <- perm[(n - n_train + 1):n]
  X_tr      <- Xstab[train_idx, , drop=FALSE]
  rownames(X_tr) <- seq_len(nrow(X_tr))

  cat(sprintf("\nFitting MacroPCA on %3.0f%% (%d cars)...\n",
              100*frac, n_train))

  fit_sub <- tryCatch(
    MacroPCA(X_tr, k=k_ref),   # no DDCpars override — raw NAs work fine
    error = function(e) { cat("  failed:", conditionMessage(e), "\n"); NULL }
  )
  if (is.null(fit_sub)) next
  if (ncol(fit_sub$loadings) < k_ref) { cat("  too few PCs\n"); next }

  signs <- align_signs(fit_sub$loadings, P_ref)

  for (j in seq_len(n_test)) {
    i   <- test_idx[j]
    car <- test_names[j]

    pred <- tryCatch(
      MacroPCApredict(Xnew = Xstab[i, , drop=FALSE],
                      InitialMacroPCA = fit_sub),
      error = function(e) NULL
    )
    if (is.null(pred) || is.null(pred$T)) next

    t_hat   <- as.numeric(pred$T) * signs
    t_ref_j <- T_ref[j, ]
    if (any(is.na(t_hat)) || any(is.na(t_ref_j))) next

    abs_d <- sqrt(sum((t_hat - t_ref_j)^2))
    norm  <- sqrt(sum(t_ref_j^2))
    rel_d <- if (norm > 1e-10) abs_d / norm else NA_real_

    results[[length(results)+1]] <- data.frame(
      Car=car, TrainFrac=frac, TrainSize=n_train,
      InTrain=(i %in% train_idx),
      AbsoluteDist=abs_d, RelativeDist=rel_d,
      stringsAsFactors=FALSE
    )
  }
}

stab_df <- if (length(results) > 0) do.call(rbind, results) else data.frame()
cat("\n=== Score-drift table ===\n")
print(stab_df, row.names = FALSE)

# 6. Plots: absolute and relative distance vs. training fraction
if (!is.null(stab_df) && nrow(stab_df) > 0) {
  p_abs <- ggplot(stab_df,
                  aes(x = 100 * TrainFrac, y = AbsoluteDist,
                      colour = Car, group = Car)) +
    geom_line() + geom_point(size = 2) +
    labs(x = "Training set size (% of original data)",
         y = expression(group("||", hat(t) - t[ref], "||")[2]),
         title = "Absolute drift of online-predicted scores",
         subtitle = "Lower = predicted scores closer to the 100%-fit reference") +
    theme_minimal()

  p_rel <- ggplot(stab_df,
                  aes(x = 100 * TrainFrac, y = RelativeDist,
                      colour = Car, group = Car)) +
    geom_line() + geom_point(size = 2) +
    labs(x = "Training set size (% of original data)",
         y = expression(group("||", hat(t) - t[ref], "||")[2] /
                        group("||", t[ref], "||")[2]),
         title = "Relative drift of online-predicted scores",
         subtitle = "Normalised by the L2 norm of the reference score vector") +
    theme_minimal()

  pdf("online_stability_study.pdf", width = 12, height = 5)
  gridExtra::grid.arrange(p_abs, p_rel, ncol = 2)
  dev.off()
  cat("\nSaved: online_stability_study.pdf\n")

  # 7. Summary across cars
  summary_df <- stab_df %>%
    group_by(TrainFrac) %>%
    summarise(
      mean_abs = mean(AbsoluteDist, na.rm = TRUE),
      mean_rel = mean(RelativeDist, na.rm = TRUE),
      max_rel  = max(RelativeDist,  na.rm = TRUE),
      .groups  = "drop"
    ) %>% arrange(desc(TrainFrac))

  cat("\n=== Mean drift across the 5 test cars ===\n")
  print(as.data.frame(summary_df), row.names = FALSE)
} else {
  cat("NOTE: stability study produced no usable rows.\n")
}


In [ ]:
# =============================================================================
# 03_contamination_analysis.R
#
# Contamination analysis on the Top Gear dataset.
# Introduces 10% cellwise outliers + 10% rowwise outliers + extra NAs,
# then evaluates ICPCA vs MacroPCA on recovery and detection.
#
# Key design decisions (matching paper Section 5 spirit):
#   - Rowwise outliers are shifted in the (k+1)-th PCA direction, which lies
#     OUTSIDE the k-dimensional subspace. This guarantees high orthogonal
#     distance OD and makes them detectable by MacroPCA.
#   - Cellwise outliers are shifted by shift_mult * column_MAD (scale-aware).
#   - Alignment fix: MacroPCA silently drops rows with >50% NAs, so its output
#     is shorter than nrow(X_cont). We map everything back to the full n rows
#     before any evaluation.
# =============================================================================

library(cellWise)
library(robustHD)
library(ggplot2)
library(ggrepel)
library(dplyr)
library(gridExtra)

set.seed(2025)

# =============================================================================
# 0. Load and preprocess Top Gear  (identical to script 02)
# =============================================================================
topgear <- NULL
tryCatch({
  data("topgear", package = "robustHD", envir = environment())
  topgear <- get("topgear", envir = environment())
}, error = function(e) invisible(NULL))
if (is.null(topgear)) {
  tryCatch({
    data("TopGear", package = "robustHD", envir = environment())
    topgear <- get("TopGear", envir = environment())
  }, error = function(e) invisible(NULL))
}
if (is.null(topgear)) {
  pkg_data     <- system.file("data", package = "robustHD")
  rda_files    <- list.files(pkg_data, pattern = "(?i)topgear",
                             full.names = TRUE, perl = TRUE)
  if (length(rda_files) > 0) {
    env_tmp <- new.env(); load(rda_files[1], envir = env_tmp)
    topgear <- get(ls(env_tmp)[1], envir = env_tmp)
  }
}
if (is.null(topgear)) stop("Could not load topgear dataset.")

car_names <- rownames(topgear)
cont_vars <- c("Price","Displacement","BHP","Torque","Acceleration",
               "TopSpeed","MPG","Weight","Length","Width","Height")
available <- intersect(cont_vars, colnames(topgear))
X_raw     <- as.matrix(topgear[, available, drop = FALSE])
log_vars  <- intersect(c("Price","Displacement","BHP","Torque","TopSpeed"),
                       available)
X_clean   <- X_raw
for (v in log_vars) X_clean[, v] <- log(X_raw[, v])

n <- nrow(X_clean)
p <- ncol(X_clean)
k <- 2

cat("=== Top Gear (log-transformed): n =", n, ", p =", p,
    ", NAs =", sum(is.na(X_clean)), "===\n\n")

# =============================================================================
# 1. Ground-truth fits on clean data
# =============================================================================
cat("Fitting ground-truth models on clean data...\n")
fit_clean_macro <- MacroPCA(X_clean, k = k)
fit_clean_icpca <- ICPCA(X_clean, k = k)

# (k+1)-th loading: direction OUTSIDE the k-dim subspace.
# We need this to contaminate rows so they have high OD.
# Fit k+1 components and take the last one.
fit_clean_k3    <- MacroPCA(X_clean, k = k + 1)
v_orth          <- fit_clean_k3$loadings[, k + 1]   # p-vector, orthog. to first k PCs

# Typical spread in the orthogonal direction (for scaling the shift)
scores_orth     <- (X_clean - matrix(fit_clean_macro$center, n, p, byrow = TRUE)) %*% v_orth
orth_scale      <- mad(scores_orth, na.rm = TRUE)
cat("Orthogonal-direction scale (mad):", round(orth_scale, 4), "\n\n")

# Subspace angle helper
subspace_angle <- function(P1, P2) {
  proj1 <- P1 %*% solve(t(P1) %*% P1) %*% t(P1)
  proj2 <- P2 %*% solve(t(P2) %*% P2) %*% t(P2)
  norm(proj1 - proj2, type = "F") / sqrt(2 * ncol(P1))
}

# =============================================================================
# 2. Contamination
#
# Rowwise:  shift rows by shift_row * orth_scale in the v_orth direction.
#           This guarantees high OD (they're off the PCA subspace).
# Cellwise: replace cells with median ± shift_cell * column_MAD.
# Missing:  add extra NAs on top of existing ones.
# =============================================================================
contaminate_topgear <- function(X,
                                frac_cell  = 0.10,
                                frac_row   = 0.10,
                                frac_miss  = 0.05,
                                shift_cell = 8,
                                shift_row  = 10) {
  Xc         <- X
  true_cells <- matrix(FALSE, nrow(X), ncol(X))
  true_rows  <- rep(FALSE, nrow(X))

  center_X <- apply(X, 2, median, na.rm = TRUE)
  col_mads <- apply(X, 2, mad,    na.rm = TRUE)
  col_mads[col_mads < 1e-10] <- 1

  # (a) Rowwise outliers: shift in orthogonal direction (off the PCA subspace)
  bad_rows <- sample(seq_len(nrow(X)), round(frac_row * nrow(X)))
  noise_sd <- 0.5 * col_mads
  for (i in bad_rows) {
    # Base point = center + large shift in v_orth + small in-plane noise
    in_plane_noise <- rnorm(ncol(X)) * noise_sd
    Xc[i, ]        <- center_X +
                      shift_row * orth_scale * v_orth +
                      in_plane_noise
    true_rows[i]   <- TRUE
  }

  # (b) Cellwise outliers: only in NON-rowwise rows, and only observed cells
  non_bad <- which(!true_rows)
  eligible_cells <- which(!is.na(X[non_bad, , drop = FALSE]))
  n_cell  <- min(round(frac_cell * length(X)), length(eligible_cells))
  cell_sel <- sample(eligible_cells, n_cell)

  # Convert eligible-subset indices back to full-matrix indices
  eligible_flat <- which(
    matrix(!true_rows, nrow(X), ncol(X)) & !is.na(X)
  )
  cell_flat <- eligible_flat[cell_sel]

  signs   <- sample(c(-1L, 1L), length(cell_flat), replace = TRUE)
  col_idx <- ceiling(cell_flat / nrow(X))          # column-major indexing
  row_idx <- ((cell_flat - 1L) %% nrow(X)) + 1L

  for (ii in seq_along(cell_flat)) {
    r <- row_idx[ii]; cc <- col_idx[ii]
    Xc[r, cc]         <- center_X[cc] + signs[ii] * shift_cell * col_mads[cc]
    true_cells[r, cc] <- TRUE
  }

  # (c) Extra missing values (MCAR, on non-already-missing cells)
  n_miss    <- round(frac_miss * nrow(X) * ncol(X))
  avail     <- which(!is.na(Xc))
  if (length(avail) > 0) {
    miss_idx  <- sample(avail, min(n_miss, length(avail)))
    Xc[miss_idx] <- NA
  }

  list(X_cont    = Xc,
       true_cells = true_cells,
       true_rows  = true_rows,
       bad_rows   = bad_rows)
}

cont   <- contaminate_topgear(X_clean)
X_cont <- cont$X_cont

cat("=== Contamination summary ===\n")
cat("  Cellwise outliers:", sum(cont$true_cells),
    sprintf("cells (%.1f%%)\n", 100 * mean(cont$true_cells)))
cat("  Rowwise outliers: ", sum(cont$true_rows),
    sprintf("rows  (%.1f%%)\n", 100 * mean(cont$true_rows)))
cat("  Missing values:   ", sum(is.na(X_cont)),
    sprintf("cells (%.1f%%)\n\n", 100 * mean(is.na(X_cont))))

# =============================================================================
# 3. Fit methods on contaminated data
# =============================================================================
cat("Fitting MacroPCA on contaminated data...\n")
fit_cont_macro <- MacroPCA(X_cont, k = k)

# =============================================================================
# 4. ALIGNMENT FIX
#
# MacroPCA silently drops rows with >50% NAs. Its output vectors/matrices
# (indrows, indcells, OD, SD, stdResid) therefore have fewer than n rows.
# We MUST map them back to the full n-row space before any evaluation.
# =============================================================================

# Rows MacroPCA actually analyzed (those with <=50% NAs in contaminated data)
frac_na_cont   <- rowMeans(is.na(X_cont))
kept_macro     <- which(frac_na_cont <= 0.5)
n_kept         <- length(kept_macro)

cat("\nAlignment: MacroPCA analyzed", n_kept, "of", n, "rows\n")
cat("Dropped rows (>50% NA):", setdiff(seq_len(n), kept_macro), "\n\n")

# Verify lengths match before expanding
stopifnot(length(fit_cont_macro$OD) == n_kept)

# Expand all MacroPCA outputs to full n-row space
OD_full       <- rep(NA_real_, n)
SD_full       <- rep(NA_real_, n)
indrows_full  <- rep(FALSE, n)
indcells_full <- matrix(0L, n, p)
stdResid_full <- matrix(NA_real_, n, p)

# ICPCA: must be fit on the SAME kept_macro rows (imputed), so that its
# scores matrix has exactly n_kept rows and aligns with kept_macro.
cat("Fitting ICPCA on contaminated data...\n")
X_cont_icpca <- X_cont[kept_macro, , drop=FALSE]
for (j in seq_len(p)) {
  med_j <- median(X_cont_icpca[, j], na.rm=TRUE)
  if (!is.finite(med_j)) med_j <- 0
  X_cont_icpca[is.na(X_cont_icpca[, j]), j] <- med_j
  if (sd(X_cont_icpca[, j], na.rm=TRUE) < 1e-10)
    X_cont_icpca[, j] <- X_cont_icpca[, j] + rnorm(nrow(X_cont_icpca), 0, 1e-6)
}
fit_cont_icpca <- tryCatch(
  ICPCA(X_cont_icpca, k = k),
  error = function(e) { message("ICPCA failed: ", conditionMessage(e)); NULL }
)

OD_full[kept_macro]          <- fit_cont_macro$OD
SD_full[kept_macro]          <- fit_cont_macro$SD

# Safe assignment for indrows (may be NULL or zero-length in some cellWise versions)
ir_vec <- fit_cont_macro$indrows
if (length(ir_vec) == n_kept) {
  indrows_full[kept_macro] <- ir_vec
} else {
  indrows_full[kept_macro] <- FALSE
}

# fit_cont_macro$indcells may be a logical/integer MATRIX (n_kept x p)
# OR a position-index VECTOR (which() of outlying cells in the kept submatrix)
local({
  ic <- fit_cont_macro$indcells
  n_kept <- length(kept_macro)
  ic_mat <- matrix(0L, n_kept, p)
  if (is.matrix(ic) && nrow(ic) == n_kept && ncol(ic) == p) {
    ic_mat <- as.integer(ic != 0)
    dim(ic_mat) <- c(n_kept, p)
  } else if (is.numeric(ic) || is.integer(ic)) {
    idx <- as.integer(ic)
    idx <- idx[!is.na(idx) & idx >= 1 & idx <= n_kept * p]
    if (length(idx) > 0) ic_mat[idx] <- 1L
  } else if (is.logical(ic) && length(ic) == n_kept * p) {
    ic_mat[which(ic)] <- 1L
  }
  indcells_full[kept_macro, ] <<- ic_mat
})
stdResid_full[kept_macro, ]  <- fit_cont_macro$stdResid

# ICPCA predictions — scores have n_kept rows (same as kept_macro)
Xhat_icpca_full <- matrix(NA_real_, n, p)
if (!is.null(fit_cont_icpca)) {
  Xhat_icpca_kept <- sweep(
    fit_cont_icpca$scores %*% t(fit_cont_icpca$loadings),
    2, fit_cont_icpca$center, "+"
  )
  # Xhat_icpca_kept has exactly n_kept rows -> maps to kept_macro
  Xhat_icpca_full[kept_macro, ] <- Xhat_icpca_kept
}

# NA-imputed X for ICPCA (replace NAs with ICPCA fitted values)
X_naimp_icpca          <- X_cont
for (j in seq_len(p)) {
  na_j    <- is.na(X_cont[, j])
  fill_j  <- !is.na(Xhat_icpca_full[, j])
  replace <- na_j & fill_j
  X_naimp_icpca[replace, j] <- Xhat_icpca_full[replace, j]
}

# =============================================================================
# 5. Subspace recovery
# =============================================================================
angle_macro <- subspace_angle(fit_cont_macro$loadings, fit_clean_macro$loadings)
angle_icpca <- if (!is.null(fit_cont_icpca))
  subspace_angle(fit_cont_icpca$loadings, fit_clean_icpca$loadings) else NA_real_

cat("=== Subspace recovery (lower = better) ===\n")
cat(sprintf("  ICPCA    angle: %.4f\n", angle_icpca))
cat(sprintf("  MacroPCA angle: %.4f\n", angle_macro))
cat(sprintf("  Improvement:    %.1f%%\n\n",
            100 * (angle_icpca - angle_macro) / max(angle_icpca, 1e-10)))

# =============================================================================
# 6. Outlier detection evaluation  (using FULL n-row aligned vectors)
# =============================================================================

# --- Rowwise ---
detected_rows <- indrows_full        # length n, FALSE for dropped rows
true_rows     <- cont$true_rows      # length n

# Only evaluate on rows that MacroPCA actually analyzed
eval_rows <- kept_macro
TP_r <- sum( detected_rows[eval_rows] &  true_rows[eval_rows])
FP_r <- sum( detected_rows[eval_rows] & !true_rows[eval_rows])
FN_r <- sum(!detected_rows[eval_rows] &  true_rows[eval_rows])
TN_r <- sum(!detected_rows[eval_rows] & !true_rows[eval_rows])

sens_row <- TP_r / max(TP_r + FN_r, 1)
spec_row <- TN_r / max(TN_r + FP_r, 1)
prec_row <- if ((TP_r + FP_r) > 0) TP_r / (TP_r + FP_r) else NA_real_
f1_row   <- if (!is.na(prec_row) && (prec_row + sens_row) > 0)
              2 * prec_row * sens_row / (prec_row + sens_row) else NA_real_

cat("=== Rowwise detection (MacroPCA, evaluated on", length(eval_rows), "rows) ===\n")
cat("  TP:", TP_r, " FP:", FP_r, " FN:", FN_r, " TN:", TN_r, "\n")
cat(sprintf("  Sensitivity: %.3f\n", sens_row))
cat(sprintf("  Specificity: %.3f\n", spec_row))
cat(sprintf("  Precision:   %s\n",   if (is.na(prec_row)) "NA" else round(prec_row,3)))
cat(sprintf("  F1:          %s\n\n", if (is.na(f1_row))   "NA" else round(f1_row,  3)))

# --- Cellwise ---
# Evaluate only on cells that (a) MacroPCA analyzed (kept_macro rows) and
# (b) were observed (not NA) in the contaminated data
detected_cells   <- indcells_full != 0                  # n x p logical
observed         <- !is.na(X_cont)                      # n x p logical

# Restrict to kept rows for fair evaluation
eval_mask        <- matrix(FALSE, n, p)
eval_mask[kept_macro, ] <- TRUE

true_cells_eval  <- cont$true_cells & observed & eval_mask
detected_c_eval  <- detected_cells  & observed & eval_mask

TP_c <- sum( detected_c_eval &  true_cells_eval)
FP_c <- sum( detected_c_eval & !true_cells_eval)
FN_c <- sum(!detected_c_eval &  true_cells_eval)

sens_cell <- TP_c / max(TP_c + FN_c, 1)
prec_cell <- if ((TP_c + FP_c) > 0) TP_c / (TP_c + FP_c) else NA_real_
f1_cell   <- if (!is.na(prec_cell) && (prec_cell + sens_cell) > 0)
               2 * prec_cell * sens_cell / (prec_cell + sens_cell) else NA_real_

cat("=== Cellwise detection (MacroPCA / DDC) ===\n")
cat("  TP:", TP_c, " FP:", FP_c, " FN:", FN_c, "\n")
cat(sprintf("  Sensitivity: %.3f\n", sens_cell))
cat(sprintf("  Precision:   %s\n",   if (is.na(prec_cell)) "NA" else round(prec_cell,3)))
cat(sprintf("  F1:          %s\n\n", if (is.na(f1_cell))   "NA" else round(f1_cell,  3)))

# =============================================================================
# 7. Residual maps
# =============================================================================
# Display rows: true outliers + top OD rows (among kept rows)
true_any  <- which(cont$true_rows | rowSums(cont$true_cells) > 0)
top_od    <- kept_macro[order(OD_full[kept_macro], decreasing = TRUE)][1:10]
disp_rows <- unique(c(true_any, top_od))
disp_rows <- intersect(disp_rows, kept_macro)        # only kept rows have data
disp_rows <- disp_rows[seq_len(min(24, length(disp_rows)))]
disp_names <- car_names[disp_rows]

# Map disp_rows to rows within the MacroPCA output
disp_in_kept <- match(disp_rows, kept_macro)

pdf("contamination_residual_map_macropca.pdf", width = 10, height = 7)
local({
  R_sub <- fit_cont_macro$stdResid[disp_in_kept, ]
  ic    <- fit_cont_macro$indcells
  if (!is.matrix(ic)) {
    ic_mat <- matrix(0L, n_kept, p)
    idx    <- as.integer(ic)
    idx    <- idx[!is.na(idx) & idx >= 1 & idx <= n_kept * p]
    if (length(idx) > 0) ic_mat[idx] <- 1L
    ic <- ic_mat
  }
  I_sub <- ic[disp_in_kept, ]
  cellMap(R_sub,
    indcells     = which(I_sub != 0),
    rowlabels    = disp_names,
    columnlabels = colnames(X_clean),
    mTitle       = "MacroPCA \u2013 Contaminated data residual map"
  )
})
dev.off()

# ICPCA residual map
resid_icpca_c <- X_naimp_icpca - Xhat_icpca_full
col_mad_c     <- apply(resid_icpca_c, 2, function(x) mad(x, na.rm=TRUE))
col_mad_c[col_mad_c < 1e-10] <- 1
stdR_icpca_c  <- sweep(resid_icpca_c, 2, col_mad_c, "/")
indc_icpca_c  <- matrix(0L, n, p)
indc_icpca_c[!is.na(stdR_icpca_c) & stdR_icpca_c >  2.576] <-  1L
indc_icpca_c[!is.na(stdR_icpca_c) & stdR_icpca_c < -2.576] <- -1L

pdf("contamination_residual_map_icpca.pdf", width = 10, height = 7)
cellMap(stdR_icpca_c[disp_rows, ],
  indcells     = which(indc_icpca_c[disp_rows, ] != 0),
  rowlabels    = disp_names,
  columnlabels = colnames(X_clean),
  mTitle       = "ICPCA \u2013 Contaminated data residual map"
)
dev.off()
cat("Saved: contamination residual maps\n")

# =============================================================================
# 8. Outlier map coloured by ground truth
# =============================================================================
ground_truth_label <- dplyr::case_when(
  cont$true_rows                      ~ "True rowwise outlier",
  rowSums(cont$true_cells) > 0        ~ "True cellwise outlier",
  TRUE                                ~ "Regular"
)

# Use only kept rows for the plot
plot_df <- data.frame(
  SD    = SD_full[kept_macro],
  OD    = OD_full[kept_macro],
  Truth = ground_truth_label[kept_macro],
  label = car_names[kept_macro],
  stringsAsFactors = FALSE
)

cSD_plot <- fit_cont_macro$cutoffSD
cOD_plot <- fit_cont_macro$cutoffOD

p_om <- ggplot(plot_df, aes(x = SD, y = OD, colour = Truth)) +
  geom_point(size = 1.8, alpha = 0.7) +
  geom_vline(xintercept = cSD_plot, linetype = "dashed", colour = "grey40") +
  geom_hline(yintercept = cOD_plot, linetype = "dashed", colour = "grey40") +
  geom_text_repel(
    data = subset(plot_df,
                  Truth != "Regular" | SD > cSD_plot | OD > cOD_plot),
    aes(label = label), size = 2.5, max.overlaps = 15
  ) +
  scale_colour_manual(values = c(
    "Regular"               = "#BBBBBB",
    "True rowwise outlier"  = "#D62728",
    "True cellwise outlier" = "#4C8BB5"
  )) +
  labs(title    = "MacroPCA outlier map \u2013 contaminated Top Gear",
       subtitle = "Coloured by ground-truth contamination label",
       x = "Score distance (SD)", y = "Orthogonal distance (OD)",
       colour = "Ground truth") +
  theme_bw(base_size = 12) +
  theme(legend.position = "bottom")

ggsave("contamination_outlier_map.pdf", p_om, width = 7, height = 6)
cat("Saved: contamination_outlier_map.pdf\n")

# =============================================================================
# 9. Summary bar charts
# =============================================================================
angle_df <- data.frame(
  Method = c("ICPCA", "MacroPCA"),
  Angle  = c(angle_icpca, angle_macro)
)
p_angle <- ggplot(angle_df, aes(x = Method, y = Angle, fill = Method)) +
  geom_bar(stat = "identity", width = 0.5, alpha = 0.85) +
  geom_text(aes(label = round(Angle, 4)), vjust = -0.4, size = 4) +
  scale_fill_manual(values = c("ICPCA" = "#E07B54", "MacroPCA" = "#2CA02C"),
                    guide = "none") +
  labs(title    = "Subspace recovery on contaminated Top Gear",
       subtitle = "Lower angle = loadings closer to clean-data reference",
       x = NULL, y = "Subspace angle") +
  theme_bw(base_size = 12)

perf_vals <- c(sens_row, spec_row,
               ifelse(is.na(prec_row), 0, prec_row),
               ifelse(is.na(f1_row),   0, f1_row),
               sens_cell,
               ifelse(is.na(prec_cell), 0, prec_cell),
               ifelse(is.na(f1_cell),   0, f1_cell))

perf_df <- data.frame(
  Metric = c("Sensitivity","Specificity","Precision","F1",
             "Sensitivity","Precision","F1"),
  Value  = perf_vals,
  Type   = c(rep("Rowwise",4), rep("Cellwise",3))
)

p_perf <- ggplot(perf_df, aes(x = Metric, y = Value, fill = Type)) +
  geom_bar(stat = "identity", position = "dodge", alpha = 0.85, width = 0.65) +
  geom_text(aes(label = sprintf("%.2f", Value)),
            position = position_dodge(width = 0.65),
            vjust = -0.4, size = 3.5) +
  scale_fill_manual(values = c("Rowwise"="#D62728","Cellwise"="#4C8BB5")) +
  scale_y_continuous(limits = c(0, 1.15), breaks = seq(0, 1, 0.2)) +
  labs(title    = "Detection performance (MacroPCA)",
       subtitle = "10% rowwise + 10% cellwise + 5% extra NAs",
       x = NULL, y = "Rate", fill = "Outlier type") +
  theme_bw(base_size = 12) +
  theme(legend.position = "bottom")

p_summary <- gridExtra::grid.arrange(p_angle, p_perf, ncol = 2)
ggsave("contamination_summary.pdf", p_summary, width = 12, height = 5)
cat("Saved: contamination_summary.pdf\n")

# =============================================================================
# 10. Console summary
# =============================================================================
cat("\n=================================================================\n")
cat("KEY FINDINGS\n")
cat("=================================================================\n")
cat(sprintf("\nSUBSPACE RECOVERY\n"))
cat(sprintf("  ICPCA    angle = %.4f\n", angle_icpca))
cat(sprintf("  MacroPCA angle = %.4f\n", angle_macro))
impr <- 100 * (angle_icpca - angle_macro) / max(angle_icpca, 1e-10)
cat(sprintf("  MacroPCA improvement: %.1f%%\n", impr))
cat("\nROWWISE DETECTION (MacroPCA)\n")
cat(sprintf("  Sensitivity %.2f | Specificity %.2f | Precision %s | F1 %s\n",
            sens_row, spec_row,
            if (is.na(prec_row)) "NA" else sprintf("%.2f", prec_row),
            if (is.na(f1_row))   "NA" else sprintf("%.2f", f1_row)))
cat("\nCELLWISE DETECTION (MacroPCA / DDC)\n")
cat(sprintf("  Sensitivity %.2f | Precision %s | F1 %s\n",
            sens_cell,
            if (is.na(prec_cell)) "NA" else sprintf("%.2f", prec_cell),
            if (is.na(f1_cell))   "NA" else sprintf("%.2f", f1_cell)))
cat("=================================================================\n")
